In [ ]:
from matplotlib_scalebar.scalebar import ScaleBar
import networkx as nx
import math
import random 
import matplotlib.pyplot as plt
import pickle
import pandas as pd
import numpy as np
import colorsys
import matplotlib.patches as mpatches
from scipy.spatial import Delaunay

In [ ]:

N = 10
HSV_tuples = [(x*1/N, 0.75, 0.95) for x in range(N)]
RGB_tuples = map(lambda x: colorsys.hsv_to_rgb(*x), HSV_tuples)
level_color_dict = {}
for i, value in enumerate(list(RGB_tuples)): 
    level_color_dict[i+1] = [value[0], value[1], value[2]]

fig, ax = plt.subplots(1)
fig.set_figheight(4)
fig.set_figwidth(2)

y=0
for label, color in level_color_dict.items():
    ax.scatter(0, y, marker='D', s=200, color=color, edgecolors='black', zorder=2)
    ax.text(0.02, y, f"{'Level ' + str(label)}",
            verticalalignment='center', fontsize=11, zorder=3)

    y += 1

ax.axis('off')
plt.box(False)

level_color_dict[0] = [0.8, 0.8, 0.8]

In [ ]:
level_color_dict[0] = [0.8, 0.8, 0.8]

In [ ]:
key = '/Multiplex 6/MP3-058-2'

In [ ]:
registration_base_path = 'Y:/coskun-lab/Aditi/01/Results/registration' + key 

In [ ]:
cell_clustering_base_path = 'Y:/coskun-lab/Aditi/01/Results/cell_segmentation_clustering' + key

In [ ]:
pixel_clustering_base_path = 'Y:/coskun-lab/Aditi/01/Results/pixel_clustering' + key 

In [ ]:
pixel_barcode_base_path = 'Y:/coskun-lab/Aditi/01/Results/cell_pixel_barcodes' + key + '/radius100'

In [ ]:
def compute_distance_between_cells(cell_a, cell_b):
    return math.sqrt((cell_b[0] - cell_a[0])**2 + (cell_b[1] - cell_a[1])**2)

In [ ]:
threshold_level_dict = {0.1: 1, 0.2: 2, 0.3: 3, 0.4: 4, 0.5:5, 0.6:6, 0.7:7, 0.8:8, 0.9:9, 1:1}

In [ ]:
def create_barcoded_cell_graph(pixel_proportions_cells_df, coord, cell_type_color_dict, cluster_color_dict, node_size, barcode_size, ox, oy, h, w, plot_barcode_only=False, plot_barcode_level=False):
    
    #construction of graph 
    pixel_proportions_cells_region_df = pixel_proportions_cells_df[(pixel_proportions_cells_df['X'].ge(coord[0])) & (pixel_proportions_cells_df['X'].le(coord[1])) & (pixel_proportions_cells_df['Y'].ge(coord[2]))  & (pixel_proportions_cells_df['Y'].le(coord[3]))]   
    pixel_proportions_cells_region_df['X'] = pixel_proportions_cells_region_df['X'] - coord[0]
    pixel_proportions_cells_region_df['Y'] = pixel_proportions_cells_region_df['Y'] - coord[2]

    G = nx.Graph()

    points = []
    cell_indices = []
    pos_dict = {}

    for ind in pixel_proportions_cells_region_df.index:
        cell_index = pixel_proportions_cells_df['Cell ID'][ind]
        point = (pixel_proportions_cells_df['X'][ind], pixel_proportions_cells_df['Y'][ind])
        G.add_node(cell_index, pos=point)
        G.nodes[cell_index]['phenotype'] = pixel_proportions_cells_df['Cell Type'][ind]
        G.nodes[cell_index]['pixel barcode'] = pixel_proportions_cells_df.iloc[ind, :np.max(np.array(list(cluster_color_dict.keys())))]
        points.append(point)
        cell_indices.append(cell_index)
        pos_dict[cell_index] = point

    result = Delaunay(points)

    for simplex in result.simplices:
        
        G.add_edge(cell_indices[simplex[0]], cell_indices[simplex[1]])
        G.add_edge(cell_indices[simplex[1]], cell_indices[simplex[2]])
        G.add_edge(cell_indices[simplex[2]], cell_indices[simplex[0]])

    #visualization of graph

    node_c = [cell_type_color_dict[G.nodes[node]['phenotype']] for node in G.nodes()]

    fig, ax = plt.subplots()
    fig.set_figheight(h)
    fig.set_figwidth(w)

    if(plot_barcode_only):
        nx.draw_networkx(G, pos = pos_dict, with_labels=False, node_color=(0.8, 0.8, 0.8), node_size=node_size, edge_color=(0.8, 0.8, 0.8))
    else:
        nx.draw_networkx(G, pos = pos_dict, with_labels=False, node_color=node_c, node_size=node_size, edge_color=(0.8, 0.8, 0.8))
     
    if(plot_barcode_only):
        if(plot_barcode_level):
            count = 0
            for node, (x, y) in pos_dict.items():
                
                offset_x = -np.ceil(len(list(cell_type_color_dict.keys()))/2)*ox
                pixel_barcode_levels = [] 
                for i, proportion in enumerate(G.nodes[node]['pixel barcode']):
                    if(proportion == 0):
                        pixel_barcode_levels.append(0)
                    else:
                        for t, level in threshold_level_dict.items():
                            if(proportion < t):
                                pixel_barcode_levels.append(level)
                                break
                  
                
                

                for level in pixel_barcode_levels: 
                    square = plt.Rectangle((x+offset_x, y-barcode_size/4*np.sqrt(2)), width=barcode_size, height=barcode_size, angle=45, fc=level_color_dict[level], fill=True, clip_on=False) 
                    plt.gca().add_patch(square)
                    offset_x+=ox
                        
                  

        else:
            for node, (x, y) in pos_dict.items():

                
                count = 0
                for i, proportion in enumerate(G.nodes[node]['pixel barcode']):
                    if(proportion !=0):
                        count+=1 


                offset_x = -np.ceil(count/2)*ox

                j = 0 

                indexed_list = [(value, index) for index, value in enumerate(G.nodes[node]['pixel barcode'])]
                sorted_indexed_list = sorted(indexed_list)
                sorted_values = [item[0] for item in sorted_indexed_list]
                original_indices_after_sort = [item[1] for item in sorted_indexed_list]

                for i, proportion in zip(original_indices_after_sort, sorted_values):
                    if(proportion !=0):
                        square = plt.Rectangle((x+offset_x, y-barcode_size/4*np.sqrt(2)), width=barcode_size, height=barcode_size, angle=45, fc=cluster_color_dict[i], fill=True, clip_on=False) 
                        plt.gca().add_patch(square)
                        offset_x+=ox
                        
                        j+=1


    else:
        if(plot_barcode_level):
            count = 0
            for node, (x, y) in pos_dict.items():
                
                offset_x = -np.ceil(len(list(cell_type_color_dict.keys()))/2)*ox
                pixel_barcode_levels = [] 
                for i, proportion in enumerate(G.nodes[node]['pixel barcode']):
                    if(proportion == 0):
                        pixel_barcode_levels.append(0)
                    else:
                        for t, level in threshold_level_dict.items():
                            if(proportion < t):
                                pixel_barcode_levels.append(level)
                                break
                  
                
                

                for level in pixel_barcode_levels: 
                    offset_y = oy
                    square = plt.Rectangle((x+offset_x, y-barcode_size/4*np.sqrt(2)+offset_y), width=barcode_size, height=barcode_size, angle=45, fc=level_color_dict[level], fill=True, clip_on=False) 
                    plt.gca().add_patch(square)
                    offset_x+=ox
        else:
            for node, (x, y) in pos_dict.items():

                count = 0
                for i, proportion in enumerate(G.nodes[node]['pixel barcode']):
                    if(proportion !=0):
                        count+=1 
                offset_x = -np.ceil(count/2)*ox

                j = 0 
                for i, proportion in enumerate(G.nodes[node]['pixel barcode']):
                    if(proportion !=0):
                        
                        offset_y = oy
                        square = plt.Rectangle((x+offset_x, y-barcode_size/4*np.sqrt(2)+offset_y), width=barcode_size, height=barcode_size, angle=45, fc=cluster_color_dict[i+1], fill=True, clip_on=False) 
                        plt.gca().add_patch(square)
                        offset_x+=ox
                        
                        j+=1

        
    plt.box(False)
    plt.show()

    return G




In [ ]:
with open(registration_base_path + '/1x/IF_scalebar_dict.pickle', 'rb') as f: 
    scalebar_dict = pickle.load(f)

In [ ]:
with open(pixel_clustering_base_path + '/cluster_color_dict.pickle', 'rb') as f:
    cluster_color_dict = pickle.load(f)

In [ ]:
import matplotlib.patches as mpatches
fig, ax = plt.subplots(1)
fig.set_figheight(6)
fig.set_figwidth(2)

y=0
for label, color in  dict(sorted(cluster_color_dict.items())).items():
    ax.scatter(0, y, marker='D', s=200, color=color, edgecolors='black', zorder=2)
    ax.text(0.02, y, f"{'Cluster ' + str(label)}",
            verticalalignment='center', fontsize=11, zorder=3)

    y += 1

ax.axis('off')
plt.box(False)

In [ ]:
with open(cell_clustering_base_path + '/1x/cell_type_color_dict.pickle', 'rb') as f: 
    cell_type_color_dict = pickle.load(f)

In [ ]:
with open(registration_base_path + '/1x/roi_coord_dict.pickle', 'rb') as f:
    roi = pickle.load(f)

In [ ]:
with open(pixel_clustering_base_path + '/wsi_labeled.pickle', 'rb') as f:
    wsi_som_clusters = pickle.load(f)

In [ ]:
pixels_outside_cells = pd.read_csv(pixel_barcode_base_path + '/pixel_proportions_outside_cells.csv').drop(columns=['Unnamed: 0'])

In [ ]:
fig, ax = plt.subplots(1, len(list(roi.keys())))
fig.set_figheight(15)
fig.set_figwidth(30)
j = 0
for name, coord in roi.items():
    img_labeled = wsi_som_clusters[coord[0]:coord[1], coord[2]:coord[3]]
    roi_scalebar = ScaleBar(scalebar_dict['Cycle 3_CH1'], 'um', length_fraction=0.25, font_properties={'size':10})
    ax[j].add_artist(roi_scalebar)
    ax[j].imshow(img_labeled, cmap=plt.cm.colors.ListedColormap(list(cluster_color_dict.values())), vmin=1, vmax=9)
    #ax[j].axis('off')
    j+=1 

In [ ]:
roi_exists = True

In [ ]:
if(roi_exists):
    with open('/'.join(pixel_barcode_base_path.split('/')[:-1]) + '/roi_for_barcode_visualization.pickle', 'rb') as f:
        roi_for_barcode_visualization = pickle.load(f)

In [ ]:
region = roi_for_barcode_visualization['A_1']

In [ ]:
pixel_barcodes = pixels_outside_cells

In [ ]:
roi_graph = create_barcoded_cell_graph(pixel_barcodes, region, cell_type_color_dict, cluster_color_dict, 80, 5, 2, 2, 10, 10, False, False)
